# 第 5 讲：回归模型与拟合诊断

> 用 scikit-learn 重构回归模型教学，输出系数、残差和拟合诊断。

## 课件导学

**任务情境**：回归模型常用于解释变量关系，本讲强调系数、误差和残差诊断必须同时出现。

**关键概念**

- 线性回归
- 训练/测试划分
- MAE 与 R2
- 残差诊断

**本讲产物**

- regression_coefficients.csv
- regression_metrics.csv
- regression_diagnostics.png

## 资料链接与阅读抓手

下面这些链接优先选官方文档或稳定资料。课前先看标题和示例，课堂中遇到参数或概念再回查。

- [scikit-learn LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)：查看线性回归模型接口。
- [scikit-learn 模型评估](https://scikit-learn.org/stable/modules/model_evaluation.html)：理解回归和分类指标的选择。
- [scikit-learn 用户指南](https://scikit-learn.org/stable/user_guide.html)：定位监督学习、聚类和降维等模块。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-05"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

## 可运行示例与讲解路线

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

n = 220
X = pd.DataFrame({
    "age": rng.integers(18, 75, n),
    "bmi": rng.normal(25, 4, n),
    "exercise_hours": rng.normal(4, 1.5, n).clip(0, None),
})
y = 65 + 0.42 * X["age"] + 1.9 * X["bmi"] - 2.5 * X["exercise_hours"] + rng.normal(0, 8, n)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)
coef = pd.DataFrame({"feature": X.columns, "coefficient": model.coef_})
metrics = pd.DataFrame({"MAE": [mean_absolute_error(y_test, pred)], "R2": [r2_score(y_test, pred)]})
coef.to_csv(OUTPUT_ROOT / "regression_coefficients.csv", index=False, encoding="utf-8-sig")
metrics.to_csv(OUTPUT_ROOT / "regression_metrics.csv", index=False, encoding="utf-8-sig")
coef, metrics

In [ ]:
residual = y_test - pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_test, pred, alpha=0.7)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[0].set_title("Observed vs predicted")
axes[0].set_xlabel("Observed")
axes[0].set_ylabel("Predicted")
axes[1].scatter(pred, residual, alpha=0.7)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_title("Residual diagnostics")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Residual")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "regression_diagnostics.png", dpi=160)
plt.show()

## 实战环节

**课堂任务**

- 替换 1 个自变量并观察 R2 变化。
- 判断残差图是否出现系统性模式。
- 写一段回归结果解释。

**课后挑战**：增加一个非线性特征或交互项，比较模型是否真正改善。

**验收清单**

- 是否明确因变量和自变量
- 是否保存系数表
- 是否用残差说明模型局限

In [ ]:
lesson_resources = [{'title': 'scikit-learn LinearRegression', 'url': 'https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html', 'reading_note': '查看线性回归模型接口。'}, {'title': 'scikit-learn 模型评估', 'url': 'https://scikit-learn.org/stable/modules/model_evaluation.html', 'reading_note': '理解回归和分类指标的选择。'}, {'title': 'scikit-learn 用户指南', 'url': 'https://scikit-learn.org/stable/user_guide.html', 'reading_note': '定位监督学习、聚类和降维等模块。'}]
lesson_deliverables = ['regression_coefficients.csv', 'regression_metrics.csv', 'regression_diagnostics.png']
lesson_checklist = ['是否明确因变量和自变量', '是否保存系数表', '是否用残差说明模型局限']

pd.DataFrame(lesson_resources).to_csv(OUTPUT_ROOT / "lesson_resources.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"deliverable": lesson_deliverables}).to_csv(OUTPUT_ROOT / "lesson_deliverables.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"check_item": lesson_checklist}).to_csv(OUTPUT_ROOT / "lesson_checklist.csv", index=False, encoding="utf-8-sig")
print("Saved courseware resource map, deliverables, and checklist.")